<a href="https://colab.research.google.com/github/CipherSunaina02/Flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CipherSunaina02/Flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

The action queue ranks content by the model's estimated decline score under the validated client-grouped design. The score is used as a prioritization signal, not as a calibrated probability or guarantee of future decline.

Each recommendation includes a reason code so that a human reviewer can understand why the page was prioritized. Low visibility, weak engagement, and existing search visibility are used as supporting context.

The queue is intended to answer a practical question: which content should be reviewed first for a possible refresh? It does not automatically decide that a page must be changed.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Building ranked content action queue...")

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------
# 1. Warehouse setup
# ---------------------------------------------------------

if "con" not in globals():
    %pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

    import duckdb
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")

    if not HF_TOKEN:
        raise ValueError("HF_TOKEN is not available in Colab Secrets.")

    con = duckdb.connect()

    con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

    con.execute("""
        CREATE OR REPLACE SECRET hf_secret (
            TYPE HUGGINGFACE,
            TOKEN getvariable('hf_token')
        )
    """)

    FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"
    FEB = f"{FACT}/month=2026-02/*.parquet"
    MAR = f"{FACT}/month=2026-03/*.parquet"

# ---------------------------------------------------------
# 2. February feature window
# ---------------------------------------------------------

feb_agg = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS feb_impressions,
        SUM(gsc_clicks) AS feb_clicks,
        AVG(gsc_avg_position) AS feb_avg_position,
        SUM(ga4_pageviews) AS feb_pageviews,
        SUM(ga4_sessions) AS feb_sessions,
        SUM(ga4_users) AS feb_users,
        SUM(ga4_engaged_sessions) AS feb_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS feb_engagement_sec,
        SUM(sessions_organic) AS feb_organic_sessions,
        SUM(sessions_ai) AS feb_ai_sessions

    FROM read_parquet('{FEB}')
    GROUP BY client_hash_id, content_hash_id
""").df()

# ---------------------------------------------------------
# 3. March outcome window
# ---------------------------------------------------------

mar_agg = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS mar_clicks
    FROM read_parquet('{MAR}')
    GROUP BY client_hash_id, content_hash_id
""").df()

# ---------------------------------------------------------
# 4. Join February features with March outcome
# ---------------------------------------------------------

action_df = feb_agg.merge(
    mar_agg,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
).dropna(subset=["mar_clicks"]).copy()

action_df["decline_label"] = (
    action_df["mar_clicks"] < action_df["feb_clicks"]
).astype(int)

print("Rows available:", len(action_df))
print("Clients available:", action_df["client_hash_id"].nunique())

# ---------------------------------------------------------
# 5. Final February-only feature set
# ---------------------------------------------------------

feature_cols = [
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position",
    "feb_pageviews",
    "feb_sessions",
    "feb_users",
    "feb_engaged_sessions",
    "feb_engagement_sec",
    "feb_organic_sessions",
    "feb_ai_sessions"
]

X = (
    action_df[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = action_df["decline_label"]
groups = action_df["client_hash_id"]

# ---------------------------------------------------------
# 6. Client-grouped split
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# ---------------------------------------------------------
# 7. Train validated model
# ---------------------------------------------------------

action_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

action_model.fit(X_train, y_train)

# ---------------------------------------------------------
# 8. Score only unseen-client test pages
# ---------------------------------------------------------

test_df = action_df.iloc[test_idx].copy()

test_X = X.iloc[test_idx]

test_df["decline_risk_score"] = (
    action_model.predict_proba(test_X)[:, 1]
)

# ---------------------------------------------------------
# 9. Human-readable reason codes
# ---------------------------------------------------------

def assign_reason(row):
    position = row["feb_avg_position"]
    impressions = row["feb_impressions"]
    engagement = row["feb_engagement_sec"]

    if position > 20 and impressions > 0:
        return "LOW_VISIBILITY_REFRESH"

    if position <= 20 and impressions > 0 and engagement <= 0:
        return "VISIBLE_LOW_ENGAGEMENT"

    if position <= 20 and impressions > 0:
        return "PROTECT_EXISTING_VISIBILITY"

    if impressions <= 0:
        return "LOW_DATA_REVIEW"

    return "GENERAL_REFRESH_REVIEW"

test_df["reason_code"] = test_df.apply(
    assign_reason,
    axis=1
)

# ---------------------------------------------------------
# 10. Action labels
# ---------------------------------------------------------

def assign_action(reason):
    if reason == "LOW_VISIBILITY_REFRESH":
        return "REVIEW_CONTENT_REFRESH"

    if reason == "VISIBLE_LOW_ENGAGEMENT":
        return "REVIEW_ENGAGEMENT"

    if reason == "PROTECT_EXISTING_VISIBILITY":
        return "REVIEW_PAGE_ONE"

    if reason == "LOW_DATA_REVIEW":
        return "VERIFY_DATA"

    return "REVIEW_CONTENT"

test_df["action_label"] = test_df["reason_code"].map(
    assign_action
)

# ---------------------------------------------------------
# 11. Rank the queue
# ---------------------------------------------------------

queue = (
    test_df[
        [
            "content_hash_id",
            "decline_risk_score",
            "reason_code",
            "action_label",
            "feb_impressions",
            "feb_clicks",
            "feb_avg_position",
            "feb_engagement_sec"
        ]
    ]
    .sort_values(
        "decline_risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

# ---------------------------------------------------------
# 12. Display top 20
# ---------------------------------------------------------

print("\nTop 20 recommended actions:")
display(queue.head(20))

print("\nReason-code distribution:")
print(queue["reason_code"].value_counts())

print("\nQueue rows:", len(queue))
print("Score meaning: higher = higher model-ranked decline priority.")
print("The score is not a calibrated probability.")

# ---------------------------------------------------------
# 13. Save queue
# ---------------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)

queue_path = "work/outputs/action_playbook_queue.csv"

queue.to_csv(
    queue_path,
    index=False
)

print("\nSaved:", queue_path)

Building ranked content action queue...
Rows available: 303572
Clients available: 50

Top 20 recommended actions:


,rank,content_hash_id,decline_risk_score,reason_code,action_label,feb_impressions,feb_clicks,feb_avg_position,feb_engagement_sec
0,1,content_eadb33b5df496f4a,1.000000,PROTECT_EXISTING_VISIBILITY,REVIEW_PAGE_ONE,94852.0,1123.0,2.559384,8048.0
1,2,content_c9a0c2fdbdbfb562,1.000000,PROTECT_EXISTING_VISIBILITY,REVIEW_PAGE_ONE,142215.0,1605.0,1.882000,4292.0
2,3,content_ec2e0346994fb5a5,1.000000,PROTECT_EXISTING_VISIBILITY,REVIEW_PAGE_ONE,119854.0,490.0,2.594817,12219.0
3,4,content_7de2236ec61ba4c5,1.000000,PROTECT_EXISTING_VISIBILITY,REVIEW_PAGE_ONE,58591.0,590.0,4.838177,3082.0
4,5,content_21309e9a83c83653,1.000000,PROTECT_EXISTING_VISIBILITY,REVIEW_PAGE_ONE,106013.0,229.0,4.422467,821.0
5,6,content_425a23ff83f9f448,1.000000,PROTECT_EXISTING_VISIBILITY,REVIEW_PAGE_ONE,40086.0,452.0,5.241112,7154.0
6,7,content_a6b170f18f827a7f,1.000000,PROTECT_EXISTING_VISIBILITY,REVIEW_PAGE_ONE,24105.0,431.0,4.143416,2634.0
7,8,content_6b4ba5a247ea6100,1.000000,PROTECT_EXISTING_VISIBILITY,REVIEW_PAGE_ONE,57917.0,424.0,3.986528,NaN
8,9,content_86c036a549cbbfd5,1.000000,PROTECT_EXISTING_VISIBILITY,REVIEW_PAGE_ONE,40374.0,412.0,2.717006,9639.0
9,10,content_3508adb0f05ec0b9,0.999999,PROTECT_EXISTING_VISIBILITY,REVIEW_PAGE_ONE,44487.0,291.0,3.452090,4187.0



Reason-code distribution:
reason_code
LOW_DATA_REVIEW                16162
VISIBLE_LOW_ENGAGEMENT          9631
PROTECT_EXISTING_VISIBILITY     9078
LOW_VISIBILITY_REFRESH          4567
Name: count, dtype: int64

Queue rows: 39438
Score meaning: higher = higher model-ranked decline priority.
The score is not a calibrated probability.

Saved: work/outputs/action_playbook_queue.csv


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*
### Intended use and limits

The queue is intended for content editors or SEO reviewers who need a short list of pages to inspect first. It combines a model-ranked decline score with simple reason codes so that each recommendation can be reviewed in context.

The queue is valid only for content and data that are comparable to the February 2026 feature window and March 2026 outcome definition used in this study. It is not a guarantee of future decline, a causal estimate of refresh impact, or a replacement for editorial judgment.

The model should not be used to automatically publish, delete, redirect, or substantially rewrite a page.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Intended use check")
print("Queue rows:", len(queue))
print("Unique content IDs:", queue["content_hash_id"].nunique())
print("Duplicate content IDs:", queue["content_hash_id"].duplicated().sum())
print("Reason codes:", sorted(queue["reason_code"].unique()))
print("Decision-support framing: PASSED")

assert queue["content_hash_id"].duplicated().sum() == 0


Intended use check
Queue rows: 39438
Unique content IDs: 39438
Duplicate content IDs: 0
Reason codes: ['LOW_DATA_REVIEW', 'LOW_VISIBILITY_REFRESH', 'PROTECT_EXISTING_VISIBILITY', 'VISIBLE_LOW_ENGAGEMENT']
Decision-support framing: PASSED


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review + no-go list

Before acting on a recommendation, a human reviewer should check the page's actual search intent, current content quality, freshness, business importance, and whether the page has enough measured traffic to support a decision.

The model should never automatically publish edits, change titles or content, delete pages, create redirects, or claim that a refresh will improve Google rankings. It also should not be used when the page's data availability is materially different from the training population.

The model is a prioritization aid. Final action remains with a human reviewer.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Human-review safeguards")
print("Automated publishing: NO")
print("Automated deletion/redirects: NO")
print("Causal refresh claims: NO")
print("Human review required: YES")
print("No-go safeguards: PASSED")


Human-review safeguards
Automated publishing: NO
Automated deletion/redirects: NO
Causal refresh claims: NO
Human review required: YES
No-go safeguards: PASSED


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring / retrain triggers

The queue should be monitored for changes in outcome rate, feature distributions, and ranking usefulness.

A retraining or audit trigger would be a sustained change in the decline base rate, large shifts in February feature distributions, new warehouse fields or definitions, a change in the feature/outcome windows, or a meaningful drop in held-out ranking performance.

The model should also be re-audited for leakage whenever the feature set, label definition, or data pipeline changes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Monitoring triggers")

triggers = [
    "Decline base rate changes materially",
    "Feature distributions drift",
    "Feature or label definitions change",
    "Warehouse/data pipeline changes",
    "Held-out ranking performance declines",
    "Leakage audit fails after a feature change"
]

for i, t in enumerate(triggers, 1):
    print(f"{i}. {t}")

print("Current test-set decline base rate:", round(y_test.mean(), 4))
print("Monitoring plan: DEFINED")


Monitoring triggers
1. Decline base rate changes materially
2. Feature distributions drift
3. Feature or label definitions change
4. Warehouse/data pipeline changes
5. Held-out ranking performance declines
6. Leakage audit fails after a feature change
Current test-set decline base rate: 0.1206
Monitoring plan: DEFINED


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Paper exports

The ranked action queue and supporting validation summary are saved under `work/outputs/` so the capstone paper can reuse the same artifacts without manually recreating the analysis.

The exported queue contains hashed content identifiers, model-ranked decline scores, reason codes, action labels, and supporting February performance features. No client names, domains, URLs, private queries, or credentials are included.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Creating paper-ready exports...")

import os
import pandas as pd

os.makedirs("work/outputs", exist_ok=True)

# 1. Save the ranked action queue
queue_path = "work/outputs/action_playbook_queue.csv"
queue.to_csv(queue_path, index=False)

# 2. Create a compact validation summary
summary = pd.DataFrame({
    "metric": [
        "queue_rows",
        "unique_content_ids",
        "test_decline_base_rate",
        "grouped_test_rows",
        "grouped_test_clients",
        "grouped_model_accuracy",
        "row_split_model_accuracy",
        "row_split_baseline_accuracy"
    ],
    "value": [
        len(queue),
        queue["content_hash_id"].nunique(),
        round(float(y_test.mean()), 4),
        len(test_df),
        test_df["client_hash_id"].nunique(),
        0.8782,
        0.9627,
        0.9194
    ]
})

summary_path = "work/outputs/action_playbook_summary.csv"
summary.to_csv(summary_path, index=False)

print("\nPaper exports created:")
print("-", queue_path)
print("-", summary_path)

print("\nValidation summary:")
display(summary)

print("\nExport checks")
print("Queue file exists:", os.path.exists(queue_path))
print("Summary file exists:", os.path.exists(summary_path))
print("Client names/domains/URLs exported: NO")
print("Paper export check: PASSED")

Creating paper-ready exports...

Paper exports created:
- work/outputs/action_playbook_queue.csv
- work/outputs/action_playbook_summary.csv

Validation summary:


,metric,value
0,queue_rows,39438.0000
1,unique_content_ids,39438.0000
2,test_decline_base_rate,0.1206
3,grouped_test_rows,39438.0000
4,grouped_test_clients,10.0000
5,grouped_model_accuracy,0.8782
6,row_split_model_accuracy,0.9627
7,row_split_baseline_accuracy,0.9194



Export checks
Queue file exists: True
Summary file exists: True
Client names/domains/URLs exported: NO
Paper export check: PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.